# SwarmSight Web - Colab Processing Backend

This notebook processes videos using configurations exported from [SwarmSight Web](https://github.com/FluentFlier/swarmsight-web).

**Workflow:**
1. Configure your sensor position and filters in the SwarmSight web tool
2. Export the config JSON file
3. Upload your videos to Google Drive
4. Update the video paths in the config to match your Drive paths
5. Run this notebook to process all videos
6. Import the resulting CSVs back into the web tool for visualization

In [ ]:
# Cell 1: Install dependencies
!pip install -q opencv-python-headless numba scipy pandas matplotlib tqdm

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully!')

In [ ]:
# Cell 3: Upload or select config.json
import json
from google.colab import files

# Option A: Upload config file
print('Upload your SwarmSight config.json file:')
uploaded = files.upload()
config_filename = list(uploaded.keys())[0]

with open(config_filename) as f:
    config = json.load(f)

print(f'\nLoaded config: module={config["module"]}, {len(config["videos"])} video(s)')
for v in config['videos']:
    print(f'  - {v["path"]} ({v.get("totalFrames", "?")} frames)')

In [ ]:
# Cell 4: Run batch processing
# First, download the processing library
!wget -q https://raw.githubusercontent.com/FluentFlier/swarmsight-web/master/colab/swarmsight_core.py -O swarmsight_core.py

from swarmsight_core import BatchRunner, validate_config

# Validate config
errors = validate_config(config)
if errors:
    print('Config errors:')
    for e in errors:
        print(f'  - {e}')
else:
    print('Config valid. Starting processing...\n')
    runner = BatchRunner(config_filename)
    output_paths = runner.run()
    print(f'\n=== Done! Generated {len(output_paths)} CSV files ===')

In [ ]:
# Cell 5: Preview results
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from IPython.display import display

if output_paths:
    csv_path = output_paths[0]
    df = pd.read_csv(csv_path)
    print(f'Results: {len(df)} rows, {len(df.columns)} columns')
    display(df.head(10))

    # Plot activity
    fig, ax = plt.subplots(figsize=(14, 4))

    if config['module'] == 'motion_analyzer':
        ax.plot(df['Frame'], df['Changed Pixels'], linewidth=0.5, color='#3b82f6')
        ax.set_ylabel('Changed Pixels')
    else:
        if 'LeftFlagellumTip-X' in df.columns:
            ax.plot(df['Frame'], df['LeftFlagellumTip-X'], linewidth=0.5, label='Left X', color='#22c55e')
            ax.plot(df['Frame'], df['RightFlagellumTip-X'], linewidth=0.5, label='Right X', color='#ef4444')
            ax.legend()
        ax.set_ylabel('Tip Position (px)')

    ax.set_xlabel('Frame')
    ax.set_title('SwarmSight Results')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 6: Summary statistics
if output_paths:
    print('=== Summary Statistics ===')
    print(f'\nVideo: {config["videos"][0]["path"]}')
    print(f'Frames processed: {len(df)}')
    print(f'\nColumn statistics:')
    numeric_cols = df.select_dtypes(include='number').columns
    display(df[numeric_cols].describe().round(2))